# Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Data Understanding

Here are some steps to understand data:
- Data Loading
- Univariate Exploratory Data Analysis
- Data Preprocessing

# ==> Data Loading

In this section, the dataset will be read directly from the dataset folder that has been downloaded through [Book Recommendation Dataset](https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset/data). As explained earlier, there are three dataset files inside the folder, namely Books, Ratings, and Users, which will be used for the model development process.

## Load the dataset

Read the dataset using the pandas.read_csv function. Implement the following code.

In [ ]:
books = pd.read_csv("/kaggle/input/book-recommendation-dataset/Books.csv")
ratings = pd.read_csv("/kaggle/input/book-recommendation-dataset/Ratings.csv")
users = pd.read_csv("/kaggle/input/book-recommendation-dataset/Users.csv")

In [ ]:
# dataset books
books.head()

In [ ]:
# dataset ratings
ratings.head()

In [ ]:
# dataset users
users.head()

In [ ]:
print('Number of book data:', len(books.ISBN.unique()))
print('Total book rating data from readers:', len(ratings.ISBN.unique()))
print('Amount of user data:', len(users['User-ID'].unique()))

Based on the output, the information obtained is as follows:

The variable "books" has 271,360 types of books and consists of 8 columns, namely:
- ISBN: a unique book identification number.
- Book-Title: the title of the book.
- Book-Author: the name of the book author.
- Year-Of-Publication: the year of publication of the book.
- Publisher: the name of the book publisher.
- Image-URL-S: the URL link for small-sized images.
- Image-URL-M: the URL link for medium-sized images.
- Image-URL-L: the URL link for large-sized images.

The variable "ratings" has 340,556 ratings for books and consists of 3 columns, namely:
- User-ID: a unique code for anonymous users who provide ratings.
- ISBN: the book identification number.
- Book-Rating: the rating given to the book.

The variable "users" has 278,858 anonymous user names and consists of 3 columns, namely:
- User-ID: a unique code for anonymous user names.
- Location: the location of the user's residence.
- Age: the age of the user.

# ==> Univariate Exploratory Data Analysis

In this stage, an analysis and exploration will be conducted on each variable to understand the distribution and individual characteristics of each variable. This understanding will later help in determining the approach or algorithm suitable for application to the data. The variables in the Book Recommendation Dataset are as follows:
- books: contains information about books.
- ratings: represents the ratings given to books by users or readers.
- users: provides user information, including demographic information.

## Book Variable

In [ ]:
# cek informasi dataset
books.info()

Based on the output, it is known that the "books.csv" file has 271,360 entries and consists of 8 columns: ISBN, Book-Title, Book-Author, Year-Of-Publication, Publisher, Image-URL-S, Image-URL-M, and Image-URL-L. It is also observed that the 'Year-Of-Publication' column is of the object data type, while publication years typically have the integer data type. Therefore, a data type correction will be performed first.

Note that when running the following code:
```
books['Year-Of-Publication'].astype('int')
```

There is an error, specifically ValueError: invalid literal for int() with base 10: 'DK Publishing Inc', indicating that there is a value in 'Year-Of-Publication' that is 'DK Publishing Inc'. It seems there is an input error, so the text values will be removed before converting it to the integer data type. Based on the investigation, there are 2 text values: 'DK Publishing Inc' and 'Gallimard'.

In [ ]:
books[(books['Year-Of-Publication'] == 'DK Publishing Inc') | (books['Year-Of-Publication'] == 'Gallimard')]

Removing values in 'Year-Of-Publication' that are text.

In [ ]:
temp = (books['Year-Of-Publication'] == 'DK Publishing Inc') | (books['Year-Of-Publication'] == 'Gallimard')
books = books.drop(books[temp].index)
books[(books['Year-Of-Publication'] == 'DK Publishing Inc') | (books['Year-Of-Publication'] == 'Gallimard')]

Changing the data type of 'Year-Of-Publication'.

In [ ]:
books['Year-Of-Publication'] = books['Year-Of-Publication'].astype(int)
print(books.dtypes)

Now, the data type of 'Year-Of-Publication' has been changed to integer. Next is to remove unnecessary variables in the model development process. Because in the content-based filtering recommendation system, recommendations will be made based on books with the same title as those read by the user, considering the book's author. Therefore, information such as image size is not needed, and the 'Image-URL-S', 'Image-URL-M', and 'Image-URL-L' features/columns can be deleted.

In [ ]:
# Removing Image-URL column of all sizes
books.drop(labels=['Image-URL-S', 'Image-URL-M', 'Image-URL-L'], axis=1, inplace=True)

books.head()

After deleting the Image-URL column, now the dataset only has 5 columns/variables left. To see how many entries there are for each variable. Run the following code.

In [ ]:
print("Number of Book ISBN numbers:", len(books['ISBN'].unique()))
print("Number of book titles:", len(books['Book-Title'].unique()))
print('Number of book authors:', len(books['Book-Author'].unique()))
print('Number of Publication Years:', len(books['Year-Of-Publication'].unique()))
print('Number of publisher names:', len(books['Publisher'].unique()))

Based on the output, the count of each variable is known. Note that the number of book titles in the dataset is 242,135, while the number of book ISBNs is 271,357. This indicates that there are some books without ISBN numbers because each book should have a unique ISBN. In this case, the dataset will be filtered to ensure that each book has a unique ISBN.

Next, a data distribution is performed to see the top 10 authors based on the number of books.

In [ ]:
# Grouping Book-Author' and count the number of books written by each author
author_counts = books.groupby('Book-Author')['Book-Title'].count()

# Sort authors in descending order
sorted_authors = author_counts.sort_values(ascending=False)

# Select the top 10 authors
top_10_authors = sorted_authors.head(10)

# The plot of the top 10 authors and the books written by the authors, then calculated using a bar plot
plt.figure(figsize=(12, 6))
top_10_authors.plot(kind='bar')
plt.xlabel('Author Name')
plt.ylabel('Number of Books')
plt.title('Top 10 Authors by Number of Books')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Based on the information above, it is known that the author with the name Agatha Christie wrote the most books, totaling more than 600 books. From this information, it is also evident that the dataset contains several authors who have written more than one book title.

## Ratings Variable

Next, exploration is conducted on the "ratings" variable, which represents the ratings given to books by readers or users. This rating dataset will be used for the model development process with collaborative filtering. Use the info() function to see information about this variable.

In [ ]:
ratings.head()

In [ ]:
ratings.info()

Based on the above output, there are a total of 1,149,780 entries and 3 columns: User-ID, which is the unique code of an anonymous user providing ratings; ISBN, which is the unique book identification number; and Book-Rating, which is the rating given to the book by the reader or user. To see the number of entries for each variable, run the following code.

In [ ]:
print('Number of User-IDs:', len(ratings['User-ID'].unique()))
print('Number of books based on ISBN:', len(ratings['ISBN'].unique()))

print('Number of book ratings:')
sorted_ratings = ratings['Book-Rating'].value_counts().sort_index()
pd.DataFrame({'Book-Rating': sorted_ratings.index, 'Sum': sorted_ratings.values})

Based on the above output, it is known that there are 105,283 users who gave book ratings. The number of books based on ISBN receiving ratings is 340,556 books, and the ratings given by each book range from 0 to 10, where 0 is the lowest rating and 10 is the highest rating.

As seen in the previous information, the "ratings" dataset has 1,149,780 rows of data, which is a substantial amount. Later, this rating dataset will be used in the model development process with collaborative filtering. Therefore, to save memory allocation during model training, not all of the rating datasets will be used. Only the first 5000 data points (excluding data point 5000) will be selected. This dataset will be used for model development with collaborative filtering because it requires user rating data to provide book title recommendations to other users. For ease and to avoid confusion with similar features, the variable name is changed to "df_rating."

In [ ]:
df_rating = ratings[:20000]
df_rating

## Users Variable

The last variable to be explored is the "users" variable. This variable contains information about anonymous users and their demographics. Use the info() function to see information about the variable.

In [ ]:
users.head()

In [ ]:
users.info()

Based on the information above, there are 278,858 entries and 3 variables: User-ID, which is the unique code for anonymous users; Location, which is the user's location; and Age, which is the user's age. It is also noted that there are some users whose age is not known. User data is useful when creating a recommendation system based on user demographics or social conditions. However, for this case study, user data will not be used in the model. In model development, the data used will be from the "books" and "ratings" datasets.

# ==> Data Preprocessing

As previously known from the data understanding stage, the Book Recommendation Dataset folder consists of three separate files: books, ratings, and users. At this stage, a file merging process will be carried out to create a unified file in line with the intended model development.

## Merging Files and Determining the Total Number of Ratings

In this stage, the books and ratings files are merged to determine the total number of ratings from these various files. Implement the following code.

In [ ]:
# Merging dataframe ratings with books based on ISBN values
books = pd.merge(ratings, books, on='ISBN', how='left')
books

After the merging process, there are 7 variables with 1,149,780 rows of data. The output above only displays a few initial and final rows of the data. This dataset will be used to create the recommendation system. Next, the calculation of the total number of ratings based on ISBN is performed with the following code.

In [ ]:
books.groupby('ISBN').sum()

# Data Preparation

# ==> Data Preparation for Model Development with Content-Based Filtering

In this stage, several techniques will be applied to prepare the data, including:
- Handling missing values.
- Standardizing book types based on ISBN.

In the content-based filtering recommendation system to be developed, each ISBN represents a unique book title, meaning that the ISBN of each book is unique. Therefore, the data needs to be prepared in advance for use in the model training process.

## Handling Missing Value

After the file merging process, the next step is to check whether there are missing values. Execute the following code.

In [ ]:
# Checking missing value using isnull() function
books.isnull().sum()

There are many missing values in most features. Only the User-ID, ISBN, and Book-Rating features have 0 missing values. The largest number of missing values is in the 'Publisher' feature, which is 118,650. 118,650 out of the total dataset of 1,149,780 is a relatively small or insignificant amount. Therefore, for this case, the missing values will be dropped, and a new variable named 'all_books_clean' will be created.

In [ ]:
all_books_clean = books.dropna()
all_books_clean

Now, the dataset consists of 1,031,128 rows. To ensure there are no more missing values in the data, run the following code.

In [ ]:
all_books_clean.isnull().sum()

Now, the dataset is clean and ready for the next step.

## Standardizing Book Types Based on ISBN

Before entering the modeling stage, it is necessary to standardize book titles based on their ISBN. If there are the same ISBNs for more than one book title, it can introduce bias into the data. Therefore, it must be ensured that there is only one ISBN for one book title.

Firstly, recheck of the data after the cleaning process in the previous stage is conducted. Create a new variable named 'fix_books' to store the dataframe.

In [ ]:
# Sort books by ISBN then put them in the fix_books variable
fix_books = all_books_clean.sort_values('ISBN', ascending=True)
fix_books

There are 1,031,128 rows of data. To check the number of ISBNs covering this data, run the following code.

In [ ]:
len(fix_books['ISBN'].unique())

Next, check the number of book titles with the following code.

In [ ]:
len(fix_books['Book-Title'].unique())

Based on the above information, it is known that the number of ISBNs does not match the number of book titles, meaning that there are ISBNs that are the same for more than one book title. This issue needs to be addressed by transforming the dataset into unique data, ready for the modeling process. Therefore, a process of removing duplicate data in the 'ISBN' column is required, and it is saved in a new variable named 'preparation'. Implement the following code.

In [ ]:
preparation = fix_books.drop_duplicates('ISBN')
preparation

After that, we perform a check again on the number of data for ISBN, book title (Book-Title), and book author's name (Book-Author). Perform the process of converting the data series into a list using the tolist() function from the library. Implement the following code.

In [ ]:
# convert the 'ISBN' data series into list form
isbn_id = preparation['ISBN'].tolist()

# convert the 'Book-Title' data series into list form
book_title = preparation['Book-Title'].tolist()

# convert the 'Book-Author' data series into list form
book_author = preparation['Book-Author'].tolist()

# convert the 'Year-Of-Publication' data series into list form
year_of_publication = preparation['Year-Of-Publication'].tolist()

# convert the 'Publisher' data series into list form
publisher = preparation['Publisher'].tolist()

print(len(isbn_id))
print(len(book_title))
print(len(book_author))
print(len(year_of_publication))
print(len(publisher))

Based on the above output, it is known that now the number of data for ISBN, book title, book author's name, year of publication, and publisher are the same or already unique data. The dataset now only has 270,144 rows of data after the duplicate value removal process. The next step is to create a dictionary to determine key-value pairs for the isbn_id, book_title, book_author, year_of_publication, and publisher data that have been prepared earlier for the development of the content-based filtering recommendation system model.

In [ ]:
books_new = pd.DataFrame({
    'isbn': isbn_id,
    'book_title': book_title,
    'book_author': book_author,
    'year_of_publication': year_of_publication,
    'publisher': publisher

})

books_new

Since the dataset is too large, and the memory allocation used will be substantial for processing the entire data in model development, for this project, only the first 20,000 data points will be used (excluding data point 20,000).

In [ ]:
books_new = books_new[:20000]

In [ ]:
books_new

This is the data that will be used in the model development process using content-based filtering techniques.

# ==> Data Preparation for Model Development with Collaborative Filtering

In the development model with collaborative filtering, the data will be divided into training and validation data in the model training process. Before splitting it into training and validation data, the data must be prepared. The rating data needs to be transformed into a numerical matrix to facilitate the model training process, allowing the model to easily recognize/learn the data. Before that, in this stage, several techniques will be applied to prepare the data, such as encoding the 'User-ID' and 'ISBN' features into integer indices, mapping 'User-ID' and 'ISBN' to the related dataframe, and finally checking some aspects of the data, such as the number of users, the number of books, and converting rating values to float for use in the model training process.

Firstly, the process of encoding the 'User-ID' and 'ISBN' features into integer indices is performed. Apply the following code.

In [ ]:
# convert User-ID to a list without matching values
user_ids = df_rating['User-ID'].unique().tolist()
print('list userIDs: ', user_ids)

# perform User-ID encoding
user_to_user_encoded = {x: i for i, x in enumerate(user_ids)}
print('encoded userID: ', user_to_user_encoded)

# carry out the process of encoding numbers into User-ID
user_encoded_to_user = {i: x for i, x in enumerate(user_ids)}
print('encoded number to userID: ', user_encoded_to_user)

Next, perform the same process for 'ISBN'.

In [ ]:
# convert ISBNs to a list without matching values
isbn_id = df_rating['ISBN'].unique().tolist()

# perform ISBN encoding
isbn_to_isbn_encoded = {x: i for i, x in enumerate(isbn_id)}

# carry out the process of encoding numbers to ISBN
isbn_encoded_to_isbn = {i: x for i, x in enumerate(isbn_id)}

After that, map the User-ID and ISBN to the related dataframe.

In [ ]:
# Disable the SettingWithCopyWarning warning
pd.options.mode.chained_assignment = None # "warn" or "raise" to turn it back on

# Mapping User-ID to user dataframe
df_rating['user'] = df_rating['User-ID'].map(user_to_user_encoded)

# Mapping ISBN to book title dataframe
df_rating['book_title'] = df_rating['ISBN'].map(isbn_to_isbn_encoded)

Check some aspects of the data, such as the number of users, the number of book titles, and convert the rating values to float.

In [ ]:
# get the number of users
num_users = len(user_to_user_encoded)
print(num_users)

# get the number of book titles
num_book_title = len(isbn_to_isbn_encoded)
print(num_book_title)

# convert the rating to a float value
df_rating['Book-Rating'] = df_rating['Book-Rating'].values.astype(np.float32)

# minimum rating value
min_rating = min(df_rating['Book-Rating'])

# maximum rating value
max_rating = max(df_rating['Book-Rating'])

print('Number of Users: {}, Number of Books: {}, Min Rating: {}, Max Rating: {}'.format(
     num_users, num_book_title, min_rating, max_rating
))

The data preparation stage is complete. The data is now ready to be used in the process of splitting into training and validation data in the collaborative filtering model development process.

# Modeling

# ==> Model Development with Content-Based Filtering

In this stage, a model will be developed using the Content-Based Filtering technique. Content-Based Filtering is an approach in recommendation systems that utilizes information or "content" from items or users to make recommendations. The basic idea is to match user preferences with the characteristics or content of items that the user has viewed or liked previously. For example, if a user likes or has purchased a book titled "Introduction to Machine Learning," and the book has features such as the author's name "Alex Smola," the system will search for other books with similar features and recommend them in the form of top-N recommendations to the user.

In the model development process, the search for important feature representations of each book title is performed using the TF-IDF (Term Frequency-Inverse Document Frequency) Vectorizer. The TF-IDF vectorizer is a tool used to convert text documents into vector representations based on the TF-IDF values of each word in the document. TF (Term Frequency) measures how often a word appears in a document, while IDF measures how unique or rare a word is in the entire collection of documents. This vector is then used to search for important feature representations of each book title based on the book author's name in the model developed with Content-Based Filtering. In this project, the [tfidfvectorizer()](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) function from the Sklearn library is used.

Meanwhile, to calculate the similarity degree between book titles, the cosine similarity technique is used. This method is used to measure the similarity between two vectors in a high-dimensional space. Cosine similarity measures the cosine angle between two vectors, and the smaller the angle, the greater the similarity between the vectors. In this project, the [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function from the Sklearn library is used.

Before starting the model development process with Content-Based Filtering, a recheck of the dataset is performed, and the dataframe from the previous stage is assigned to the variable "data."

In [ ]:
data = books_new
data.sample(5)

## TF-IDF Vectorizer

Import the tfidfvectorizer() function from the Sklearn library.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TfidfVectorizer
tf = TfidfVectorizer()

# Perform IDF calculations on book_author data
tf.fit(data['book_author'])

# Mapping array from integer index features to name features
tf.get_feature_names_out()

Next, fit and transform it into a matrix.

In [ ]:
# Performs a fit and then transforms it into matrix form
tfidf_matrix = tf.fit_transform(data['book_author'])

# View the tfidf matrix size
tfidf_matrix.shape

Based on the output, the matrix has a size of (20,000, 8746). The value 20,000 represents the data size, and 8746 represents the matrix of book author names. To generate the tf-idf vector in matrix form, use the todense() function.

In [ ]:
tfidf_matrix.todense()

Next, let's look at the tf-idf matrix for some book titles and author names in the form of a dataframe, where columns are filled with book author names, and rows are filled with book titles.

In [ ]:
pd.DataFrame(
    tfidf_matrix.todense(),
    columns=tf.get_feature_names_out(),
    index=data.book_title
).sample(15, axis=1).sample(10, axis=0)

The output of the tf-idf matrix successfully identifies important feature representations of each book title with the tfidfvectorizer function. In this case, the dataset is displayed as a sample data, so the entire matrix is not visible. Out of 20,000 data, only a random sample of 10 book titles on the vertical axis and 15 book author names on the horizontal axis is selected.

## Cosine Similarity

In the previous stage, the correlation between book titles and book authors has been successfully identified. Now, the process of calculating the similarity degree between book titles using the cosine similarity technique will be carried out.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculating cosine similarity on the tf-idf matrix
cosine_sim = cosine_similarity(tfidf_matrix)
cosine_sim

In this stage, the cosine similarity calculation process for the tfidf_matrix dataframe obtained in the previous stage is performed. Using the cosine_similarity function from the sklearn library, similarity values between book titles are obtained. The code above produces output in the form of a similarity matrix in array format.

Next, let's look at the similarity matrix of each book title by displaying the book title names in 5 sample columns (axis = 1) and 10 sample rows (axis = 0).

In [ ]:
# Create a dataframe from the cosine_sim variable with rows and columns in the form of book titles
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['book_title'], columns=data['book_title'])
print('Shape:', cosine_sim_df.shape)

# View the similarity matrix for each book title
cosine_sim_df.sample(5, axis=1).sample(10, axis=0)

With cosine similarity, it has successfully identified the similarity between one book title and another. The shape (20000, 20000) represents the size of the similarity matrix from the data. Based on the available data, the matrix above is actually sized 20,000 book titles x 20,000 book titles (each along the X and Y axes). This means that the system has successfully identified the similarity level for 20,000 book titles. However, it is not possible to display all the data here. Therefore, only 10 book titles on the vertical axis and 5 book titles on the horizontal axis are selected. With the similarity data obtained earlier, the system will recommend a list of book titles similar to those previously purchased or read by the user.

## Getting Recommendations

In this stage, a function named book_recommendations will be created with several parameters as follows:
- book_title: the name of the book title (index of the similarity dataframe).
- similarity_data: Dataframe about the defined similarity.
- items: Names and features used to define similarity, in this case, 'book_title' and 'book_author'.
- k: The number of top-N recommendations provided by the recommendation system. By default, k is set to 5.

Before writing the code, it is essential to remember that the definition of a recommendation system states that the system's output is in the form of top-N recommendations. Therefore, a certain number of book title recommendations need to be given to the user, as set in the parameter k.

In [ ]:
def book_recommendation(book_title, similarity_data=cosine_sim_df, items=data[['book_title', 'book_author']], k=5):
     # Retrieve data by using argpartition to partition indirectly along a given axis
     # Dataframe converted to numpy
     # Range(start, stop, step)
     index = similarity_data.loc[:,book_title].to_numpy().argpartition(range(-1, -k, -1))

     # Retrieve data with the greatest similarity from the existing index
     closest = similarity_data.columns[index[-1:-(k+2):-1]]
    
     # Drop book_title so that the name of the book you are looking for does not appear in the recommendation list
     closest = closest.drop(book_title, errors='ignore')

     return pd.DataFrame(closest).merge(items).head(k)

Note that using argpartition, the process of taking the top k values from the similarity data (in this case: cosine_sim_df dataframe) is performed. Next, retrieve data from the highest to lowest weights (similarity levels). This data is then stored in the variable closest. Furthermore, it is necessary to remove the searched book_title so that it does not appear in the recommendation list. In this case, book_title needs to be dropped to avoid appearing in the provided recommendation list.

Use the book_recommendation function to generate the top 5 book recommendations recommended by the system.

In [ ]:
book_title_test = "Entering the Silence : Becoming a Monk and a Writer (The Journals of Thomas Merton, V. 2)" # book title example

data[data.book_title.eq(book_title_test)]

Note that the book title 'Entering the Silence: Becoming a Monk and a Writer (The Journals of Thomas Merton, V. 2)' is written by 'Thomas Merton'. Now, use the book_recommendation function to get recommendations based on this book title.

In [ ]:
# Get recommendations for similar book titles
book_recommendation(book_title_test)

Based on the output above, the system successfully recommends the top 5 book titles with the book author's name ('Thomas Merton') category.

# ==> Model Development with Collaborative Filtering

In this model development process, collaborative filtering technique will be applied to create a recommendation system. This technique requires user or reader rating data. Collaborative filtering is one of the methods in recommendation systems that predicts user preferences for items based on information from other users (collaboration). The basic idea behind collaborative filtering is that users with similar preferences in the past tend to have similar preferences for items in the future. In this project, a collaborative filtering model based on user similarity (User-Based Collaborative Filtering) will be created.

The development of the collaborative filtering model in this project will result in recommendations for a number of book titles that match the user's preferences based on previously given ratings. From user rating data, names of similar book titles that the user has not read or purchased yet will be identified and recommended.

After the data preparation stage for this model development has been done in the previous data preparation section, the next step is to split the data into training and validation data, followed by the training model process. In the training process, the model calculates the match score between users and book titles using the embedding technique. First, the embedding process is performed on user and book title data. Next, perform the dot product multiplication operation between user embedding and book title. In addition, bias is added for each user and book title. The match score is set on a [0,1] scale with the sigmoid activation function. The model is created with the RecommenderNet class using the [Keras Model class](https://keras.io/api/models/model/). The code for this RecommenderNet class is inspired by a tutorial on the [Keras website](https://keras.io/examples/structured_data/collaborative_filtering_movielens/) with some adaptations to suit the current case. The model will use Binary Crossentropy to calculate the loss function, Adam (Adaptive Moment Estimation) as the optimizer, and Root Mean Squared Error (RMSE) as the evaluation metric.

## Splitting Data for Training and Validation

Before splitting the data into training and validation, the data is first shuffled to make its distribution random.

In [ ]:
df_rating = df_rating.sample(frac=1, random_state=42)
df_rating

Next, the process of dividing the data into training and validation data is carried out with a 90:10 composition. However, before that, it is necessary to map user and book title data into a single value first. Then, create ratings on a scale of 0 to 1 to facilitate the training process.

In [ ]:
# create a variable x to match user data and book title into one value
x = df_rating[['user', 'book_title']].values

# create a y variable to create a rating of the results
y = df_rating['Book-Rating'].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values

# divide into 90% train data and 10% validation data

train_indices = int(0.9 * df_rating.shape[0])
x_train, x_val, y_train, y_val = (
     x[:train_indices],
     x[train_indices:],
     y[:train_indices],
     y[train_indices:]
)

print(x, y)

Up to this point, the data is ready to be used in the collaborative filtering model development.

## Training Process

In the model training process, the model will calculate the match score between users and book titles using the embedding technique. First, the embedding process is performed on user and book title data. Next, perform the dot product multiplication operation between user embedding and book title. In addition, bias can also be added for each user and book title. The match score is set on a [0, 1] scale with the sigmoid activation function.

Here, the model is created with the RecommenderNet class using the Keras Model class. The code for this RecommenderNet class is inspired by a tutorial on the Keras website with some adaptations to suit the current case.

In [ ]:
class RecommenderNet(tf.keras.Model):

     # function initialization
     def __init__(self, num_users, num_book_title, embedding_size, dropout_rate=0.2, **kwargs):
         super(RecommenderNet, self).__init__(**kwargs)
         self.num_users = num_users
         self.num_book_title = num_book_title
         self. embedding_size = embedding_size
         self.dropout_rate = dropout_rate
        
         self.user_embedding = layers.Embedding( # user embedding layer
             num_users,
             embedding_size,
             embeddings_initializer = 'he_normal',
             embeddings_regularizer =keras.regularizers.l2(1e-6)
         )
         self.user_bias = layers.Embedding(num_users, 1) # layer embedding user bias

         self.book_title_embedding = layers.Embedding( # book_title embedding layer
             num_book_title,
             embedding_size,
             embeddings_initializer = 'he_normal',
             embeddings_regularizer =keras.regularizers.l2(1e-6)
         )
         self.book_title_bias = layers.Embedding(num_book_title, 1) # layer embedding book_title
        
         self.dropout = layers.Dropout(rate=dropout_rate)
    
     def call(self, inputs):
         user_vector = self.user_embedding(inputs[:, 0]) # call embedding layer 1
         user_vector = self.dropout(user_vector)
         user_bias = self.user_bias(inputs[:, 0]) # call embedding layer 2

         book_title_vector = self.book_title_embedding(inputs[:, 1]) # call embedding layer 3
         book_title_vector = self.dropout(book_title_vector)
         book_title_bias = self.book_title_bias(inputs[:, 1]) # call embedding layer 4

         dot_user_book_title = tf.tensordot(user_vector, book_title_vector, 2) # dot product multiplication

         x = dot_user_book_title + user_bias + book_title_bias

         return tf.nn.sigmoid(x) # activate sigmoid

Next, compile the model.

In [ ]:
model = RecommenderNet(num_users, num_book_title, 50) # initialize model

# model compile
model.compile(
    loss = tf.keras.losses.BinaryCrossentropy(),
    optimizer = keras.optimizers.Adam(learning_rate=1e-4),
    metrics = [tf.keras.metrics.RootMeanSquaredError()]
)

This model uses Binary Crossentropy to calculate the loss function, Adam (Adaptive Moment Estimation) as the optimizer, and root mean squared error (RMSE) as the metric for evaluation.

In [ ]:
# start the training process

history = model.fit(
    x = x_train,
    y = y_train,
    batch_size = 16,
    epochs = 50,
    validation_data = (x_val, y_val)
)

Based on the results of the model training process, satisfactory results are obtained, and the model converges at around 50 epochs. From this process, a Root Mean Squared Error (RMSE) value of approximately 0.2948 and an RMSE on the validation data of 0.3359 are obtained. These values are quite good for a recommendation system. To see the results of the model development, the next step is to get book title recommendations based on the developed model.

## Getting Book Title Recommendations

To obtain book title recommendations, first, a random sample of users is taken, and the variable book_not_read is defined, which is a list of books that the user has never read or purchased. This variable book_not_read will be the book titles recommended by the system.

The book_not_visited variable is obtained by using the bitwise NOT operator (~) on the book_read_by_user variable.

In [ ]:
book_df = books_new

# take a sample of users
user_id = df_rating['User-ID'].sample(1).iloc[0]
book_readed_by_user = df_rating[df_rating['User-ID'] == user_id]

# create variable book_not_readed
book_not_readed = book_df[~book_df['isbn'].isin(book_readed_by_user['ISBN'].values)]['isbn']
book_not_readed = list(
    set(book_not_readed)
    .intersection(set(isbn_to_isbn_encoded.keys()))
)

book_not_readed = [[isbn_to_isbn_encoded.get(x)] for x in book_not_readed]
user_encoder = user_to_user_encoded.get(user_id)
user_book_array = np.hstack(
    ([[user_encoder]] * len(book_not_readed), book_not_readed)
)

Next, to get book title recommendations, use the model.predict() function from the Keras library.

In [ ]:
ratings_model = model.predict(user_book_array).flatten()

top_ratings_indices = ratings_model.argsort()[-10:][::-1]

recommended_book_ids = [
    isbn_encoded_to_isbn.get(book_not_readed[x][0]) for x in top_ratings_indices
]

top_book_user = (
    book_readed_by_user.sort_values(
        by='Book-Rating',
        ascending=False
    )
    .head(10)['ISBN'].values
)

book_df_rows = book_df[book_df['isbn'].isin(top_book_user)]

# Displays book recommendations in DataFrame form
book_df_rows_data = []
for row in book_df_rows.itertuples():
    book_df_rows_data.append([row.book_title, row.book_author])

recommended_book = book_df[book_df['isbn'].isin(recommended_book_ids)]

recommended_book_data = []
for row in recommended_book.itertuples():
    recommended_book_data.append([row.book_title, row.book_author])

# Create a DataFrame for output
output_columns = ['Book Title', 'Book Author']
df_book_readed_by_user = pd.DataFrame(book_df_rows_data, columns=output_columns)
df_recommended_books = pd.DataFrame(recommended_book_data, columns=output_columns)

# Displays recommendation results in DataFrame form
print("Showing recommendation for users: {}".format(user_id))
print("===" * 9)
print("Book with high ratings from user")
print("----" * 8)
print(df_book_readed_by_user)
print("----" * 8)
print("Top 10 books recommendation")
print("----" * 8)
df_recommended_books


Based on the output above, recommendations have been successfully made to the user. The results above are recommendations for the user with the id 2288. From this output, a comparison can be made between 'Books with high ratings from the user' and 'Top 10 books recommendations' for the user.

Note that some recommended book titles also provide the name of the book author that matches the user's rating. The top 10 book recommendations along with the author's name for that user are obtained, and there is one book title that is the highest-rated book by the user.

# Evaluation

# ==> Model Evaluation with Content-Based Filtering

The metrics used for evaluating the model with content-based filtering in this project are Precision, Recall, and F1-Score. These metrics are commonly used to measure model performance. Precision is the ratio of relevant items produced by the model to the total items produced. Recall is the ratio of relevant items produced by the model to the total items that should be recommended. Meanwhile, F1 Score is a combination of Precision and Recall, providing a single value that measures the balance between the two.

Before calculating the evaluation metrics using precision, recall, and f1 score, data consisting of actual labels is needed, which is used to assess the model's prediction results; this data is called ground truth data. Ground truth data in this project is created using the cosine similarity results, where each row and column represents a book title, and the value in each cell in the dataframe represents a label. The value 1 for similar and the value 0 for not similar. It is also necessary to set a threshold value to decide whether the similarity value between two items should be considered 1 (similar) or 0 (not similar).

In [ ]:
# Determines the threshold for categorizing similarity as 1 or 0
threshold = 0.5

# Create ground truth data with threshold assumptions
ground_truth = np.where(cosine_sim >= threshold, 1, 0)

# Displays several values in the ground truth matrix
ground_truth_df = pd.DataFrame(ground_truth, index=data['book_title'], columns=data['book_title']).sample(5, axis=1).sample(10, axis=0)

In the above code, a threshold value is set to 0.5. This threshold value is adjusted according to needs and characteristics after seeing the previous recommendation results. Then, a ground truth matrix is created using the np.where() function from NumPy. This matrix will have a value of 1 in positions where the cosine similarity value between two items is greater than or equal to the set threshold value, and a value of 0 in positions where the similarity value is below the threshold. After the matrix is created, the results are presented in the form of a dataframe. Rows and columns of this ground truth DataFrame are indexed using book titles from the data. Here is a view of the ground truth dataframe.

In [ ]:
ground_truth_df

After creating the ground truth matrix containing the actual labels from cosine similarity results. Next, the process of calculating the model evaluation with precision, recall, and f1 score metrics is carried out. First, import the precision_recall_fscore_support function from the [Sklearn library](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html), which is used to calculate precision, recall, and f1 score. Because of memory allocation limitations on the device, only about 10,000 samples from the cosine similarity and ground truth matrices are taken. This is done to speed up the calculation process, especially considering the relatively large size of the matrices. Then, the cosine similarity and ground truth matrices are converted into one-dimensional arrays to facilitate comparison and metric calculation.

A threshold is also used to categorize cosine similarity values as 1 or 0. If the similarity value is above or equal to the threshold, it is considered 1 (positive), and if it is below the threshold, it is considered 0 (negative). The results are stored in the predictions array. Finally, the precision_recall_fscore_support function is used to calculate precision, recall, and f1 score. The parameter average='binary' is used because we are measuring performance in the context of binary classification (1 or 0). The parameter zero_division=1 is used to avoid division by zero if there are classes that are not present in the prediction. The implementation of the code is as follows:

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

# Takes a small portion of the cosine similarity matrix and ground truth matrix
sample_size = 10000
cosine_sim_sample = cosine_sim[:sample_size, :sample_size]
ground_truth_sample = ground_truth[:sample_size, :sample_size]

# Converts the cosine similarity matrix to a one-dimensional array for comparison
cosine_sim_flat = cosine_sim_sample.flatten()

# Converts the ground truth matrix into a one-dimensional array
ground_truth_flat = ground_truth_sample.flatten()

# Calculate evaluation metrics
predictions = (cosine_sim_flat >= threshold).astype(int)
precision, recall, f1, _ = precision_recall_fscore_support(
     ground_truth_flat, predictions, average='binary', zero_division=1
)

print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)

Based on the evaluation results, values for each evaluation metric, namely precision, recall, and F1 Score, are obtained. The Precision value is obtained as 1.0, meaning all positive model predictions are correct, and there are no false positives. The Recall value is obtained as 1.0, indicating that the model successfully identified about 100% of all truly relevant items. The F1 Score value is also around 1.0, indicating a good balance between precision and recall, and the model tends to provide very good results for both classes (positive and negative). In conclusion, based on these evaluation metric results, the model works very good in recommending items with content-based filtering.

# ==> Model Evaluation with Collaborative Filtering

As seen in the model training process in the modeling section, the metric used to evaluate the model in the Collaborative Filtering model in this project is [Root Mean Squared Error (RMSE)](https://www.statisticshowto.com/probability-and-statistics/regression-analysis/rmse-root-mean-square-error/). RMSE is a commonly used evaluation metric to measure how well a model predicts continuous values by comparing predicted values with actual values. In the context of collaborative filtering, RMSE is typically used to assess how well a collaborative model predicts user preferences for items.

Based on the results of the model training process in the modeling stage, training results in the form of RMSE information in the training and validation data are obtained. To visualize the model training process, a plotting process for the evaluation metric with matplotlib is performed. Apply the following code.

In [ ]:
plt.plot(history.history['root_mean_squared_error'])
plt.plot(history.history['val_root_mean_squared_error'])
plt.title('model_metrics')
plt.ylabel('root_mean_squared_error')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

Based on the visualization results of the RMSE evaluation metric for the developed model, it is observed that the model converges at around 50 epochs, and based on the plot of the model metrics, it provides a relatively small MSE value. From this process, a final error value of 0.2948 is obtained, and the error on the validation data is 0.3359. These values indicate a reasonably good result for the generated recommendation system. The smaller the RMSE value, the better the model is at predicting user preferences for items. This is what makes the recommendation results from the model quite accurate.